In [ ]:
import pandas as pd
import os
import glob

folders = [
    "datafiles/kinect_good_preprocessed_not_cut_A11_mediapipe",
    "datafiles/kinect_good_preprocessed_A9_mediapipe"
]

pairs = [
    ('left_shoulder', 'right_shoulder'),
    ('left_elbow', 'right_elbow'),
    ('left_wrist', 'right_wrist'),
    ('left_hip', 'right_hip'),
    ('left_knee', 'right_knee'),
    ('left_ankle', 'right_ankle')
]

for target_folder in folders:
    all_files = glob.glob(os.path.join(target_folder, "*.csv"))
    print(f"Processing folder: {target_folder} ({len(all_files)} files found)")

    count = 0
    for f in all_files:
        if "_mirrored" in f:
            continue

        df = pd.read_csv(f)
        mirrored_df = df.copy()

        x_cols = [c for c in df.columns if "_3d_x" in c]
        mirrored_df[x_cols] = 1.0 - df[x_cols]

        for left, right in pairs:
            for axis in ['x', 'y', 'z']:
                l_col = f"{left}_3d_{axis}"
                r_col = f"{right}_3d_{axis}"

                if l_col in df.columns and r_col in df.columns:
                    mirrored_df[l_col] = df[r_col]
                    mirrored_df[r_col] = df[l_col]

                    if axis == 'x':
                        mirrored_df[l_col] = 1.0 - mirrored_df[l_col]
                        mirrored_df[r_col] = 1.0 - mirrored_df[r_col]

        new_path = f.replace(".csv", "_mirrored.csv")
        mirrored_df.to_csv(new_path, index=False)
        count += 1

    print(f"Successfully created {count} mirrored files in {target_folder}.\n")

print("All folders processed.")

In [ ]:

import pandas as pd
import os
import glob
from tqdm import tqdm

cut_folder   = "datafiles/kinect_good_preprocessed_A9_mediapipe"
uncut_folder = "datafiles/kinect_good_preprocessed_not_cut_A11_mediapipe"
output_folder = "datafiles/labeled_files"

os.makedirs(output_folder, exist_ok=True)

uncut_files = glob.glob(os.path.join(uncut_folder, "*.csv"))
print(f"Found {len(uncut_files)} uncut files. Starting labeling...")

for uncut_path in tqdm(uncut_files):
    filename = os.path.basename(uncut_path)
    cut_path = os.path.join(cut_folder, filename)

    if not os.path.exists(cut_path):
        print(f"Warning: No cut file found for {filename}. Skipping.")
        continue

    df_uncut = pd.read_csv(uncut_path)
    df_cut   = pd.read_csv(cut_path)

    start_frame = df_cut['FrameNo'].min()
    end_frame   = df_cut['FrameNo'].max()

    df_uncut['activity_label'] = (
        (df_uncut['FrameNo'] >= start_frame) &
        (df_uncut['FrameNo'] <= end_frame)
    ).astype(int)

    save_path = os.path.join(output_folder, filename)
    df_uncut.to_csv(save_path, index=False)

print(f"\nDone! Labeled files saved to: {output_folder}")

In [ ]:

# This ignore list includes files which does not squat or is badly filmed disrupting the model
ignore_list = [
    "B2_kinect.csv", "B3_kinect.csv", "B4_kinect.csv",
    "B5_kinect.csv", "A130_kinect.csv"
]

experiment_variants = [
    {
        "note": "LSTM_Dense_Uni",
        "seed": 42,
        "seq_length": 5,
        "hidden_size": 128,
        "lr": 0.0005,
        "batch_size": 64,
        "epochs": 30,
        "patience": 5,
        "num_layers": 2,
        "dropout": 0.1,
        "input_size": 39,
        "use_scaling": False,
        "activation": "identity",
        "init": "default",
        "bidirectional": False,
        "rnn_type": "LSTM",
    },
]



In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import Dataset
import torch.nn.init as init
import random
import numpy as np


class ActivityGatekeeper(nn.Module):
    def __init__(self, config):
        super(ActivityGatekeeper, self).__init__()

        self.bidirectional = config.get("bidirectional", False)
        multiplier = 2 if self.bidirectional else 1

        rnn_type = config.get("rnn_type", "LSTM")

        if rnn_type == "GRU":
            self.rnn = nn.GRU(
                config["input_size"],
                config["hidden_size"],
                config["num_layers"],
                batch_first=True,
                dropout=config["dropout"] if config["num_layers"] > 1 else 0,
                bidirectional=self.bidirectional,
            )
        else:
            self.rnn = nn.LSTM(
                config["input_size"],
                config["hidden_size"],
                config["num_layers"],
                batch_first=True,
                dropout=config["dropout"] if config["num_layers"] > 1 else 0,
                bidirectional=self.bidirectional,
            )

        if config.get("activation") == "relu":
            self.act = nn.ReLU()
        elif config.get("activation") == "tanh":
            self.act = nn.Tanh()
        else:
            self.act = nn.Identity()

        self.fc = nn.Linear(config["hidden_size"] * multiplier, 1)

        if config.get("init") == "xavier":
            init.xavier_uniform_(self.fc.weight)
        elif config.get("init") == "kaiming":
            init.kaiming_normal_(self.fc.weight)

    def forward(self, x):
        out, _ = self.rnn(x)
        last_step = out[:, -1, :]
        activated = self.act(last_step)
        return self.fc(activated)


class ActivityDataset(Dataset):
    def __init__(self, file_paths, config, scaler=None):
        self.X, self.y = [], []
        seq_length = config["seq_length"]
        for path in file_paths:
            df = pd.read_csv(path)
            feat_cols = get_feat_cols(df.columns)
            X_data = df[feat_cols].values.astype('float32')
            y_data = df['activity_label'].values.astype('float32')
            if scaler is not None:
                X_data = scaler.transform(X_data)
            if len(df) < seq_length:
                continue
            for i in range(len(df) - seq_length + 1):
                self.X.append(X_data[i: i + seq_length])
                self.y.append(y_data[i + seq_length - 1])

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return torch.tensor(self.X[idx]), torch.tensor([self.y[idx]])


def get_with_mirrors(file_list):
    res = []
    for f in file_list:
        res.append(f)
        m = f.replace(".csv", "_mirrored.csv")
        if os.path.exists(m):
            res.append(m)
    return res



def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"Random seed set to: {seed}")

def compute_pos_weight(file_list, device):
    """Compute BCE pos_weight from a list of CSV files."""
    labels = []
    for f in file_list:
        labels.extend(pd.read_csv(f)['activity_label'].values)
    neg, pos = np.bincount(np.array(labels).astype(int))
    return torch.tensor([neg / pos], dtype=torch.float).to(device)

def get_feat_cols(columns):
    """Return only the 3D joint coordinate columns."""
    return [c for c in columns if '_3d_' in c]


In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
import joblib
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_path = "datafiles/labeled_files/"
originals = sorted([
    f for f in glob.glob(os.path.join(data_path, "*.csv"))
    if "_mirrored" not in f and os.path.basename(f) not in ignore_list
])

kf = KFold(n_splits=10, shuffle=True, random_state=42)

for config in experiment_variants:
    set_seed(config["seed"])

    best_config_for_run = config.copy()
    best_overall_mae    = float('inf')

    ckpt_path  = f"gatekeeper_best_{config['note']}.pth"
    scaler_path = f"scaler_best_{config['note']}.joblib"

    for p in (ckpt_path, scaler_path):
        if os.path.exists(p):
            os.remove(p)



    all_start_diffs = []
    all_stop_diffs  = []
    all_file_results = []
    for fold, (train_idx, test_idx) in enumerate(kf.split(originals)):
        print(f"\n--- Fold {fold + 1} ---")
        train_orig = [originals[i] for i in train_idx]
        test_orig  = [originals[i] for i in test_idx]
        random.shuffle(train_orig)
        split_idx   = int(len(train_orig) * 0.9)
        train_slice = train_orig[:split_idx]
        val_slice   = train_orig[split_idx:]
        scaler = None
        if config.get("use_scaling", False):
            scaler = MinMaxScaler()
            train_samples = [
                pd.read_csv(f)[get_feat_cols(pd.read_csv(f).columns)].values
                for f in train_slice
            ]
            scaler.fit(np.vstack(train_samples))
        pos_weight = compute_pos_weight(train_slice, device)
        train_loader = DataLoader(
            ActivityDataset(get_with_mirrors(train_slice), config, scaler),
            batch_size=config["batch_size"], shuffle=True
        )
        val_loader = DataLoader(
            ActivityDataset(get_with_mirrors(val_slice), config, scaler),
            batch_size=config["batch_size"], shuffle=False
        )
        model     = ActivityGatekeeper(config).to(device)
        optimizer = optim.Adam(model.parameters(), lr=config["lr"])
        criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        best_v_loss, patience_count = float('inf'), 0
        fold_ckpt = f"best_fold_{fold}_{config['note']}.pth"
        # Training loop
        for epoch in range(config["epochs"]):
            model.train()
            for x, y in train_loader:
                optimizer.zero_grad()
                criterion(model(x.to(device)), y.to(device)).backward()
                optimizer.step()
            model.eval()
            v_loss = 0.0
            with torch.no_grad():
                for x, y in val_loader:
                    v_loss += criterion(model(x.to(device)), y.to(device)).item()
            avg_v_loss = v_loss / len(val_loader)
            if avg_v_loss < best_v_loss:
                best_v_loss    = avg_v_loss
                patience_count = 0
                torch.save(model.state_dict(), fold_ckpt)
            else:
                patience_count += 1
                if patience_count >= config["patience"]:
                    break
        model.load_state_dict(torch.load(fold_ckpt))
        model.eval()
        fold_start_errs, fold_stop_errs = [], []
        for test_file in test_orig:
            df = pd.read_csv(test_file)
            actual_indices = df[df['activity_label'] == 1]['FrameNo'].values
            if len(actual_indices) == 0:
                continue
            actual_start = actual_indices[0]
            actual_stop  = actual_indices[-1]
            feat_cols = get_feat_cols(df.columns)
            X_eval    = scaler.transform(df[feat_cols].values) if scaler else df[feat_cols].values.astype('float32')
            seq_len = config["seq_length"]
            preds   = [0] * (seq_len - 1)
            with torch.no_grad():
                for i in range(len(df) - seq_len + 1):
                    window = torch.tensor(X_eval[i: i + seq_len]).float().unsqueeze(0).to(device)
                    preds.append(1 if torch.sigmoid(model(window)).item() >= 0.5 else 0)
            pred_indices = np.where(np.array(preds) == 1)[0]
            if len(pred_indices) > 0:
                p_start   = df.iloc[pred_indices[0]]['FrameNo']
                p_stop    = df.iloc[pred_indices[-1]]['FrameNo']
                start_off = abs(p_start - actual_start)
                stop_off  = abs(p_stop  - actual_stop)
            else:
                start_off = len(df)
                stop_off  = len(df)
            total_off = start_off + stop_off
            fold_start_errs.append(start_off)
            fold_stop_errs.append(stop_off)
            all_start_diffs.append(start_off)
            all_stop_diffs.append(stop_off)
            all_file_results.append({
                "filename":  os.path.basename(test_file),
                "start_off": start_off,
                "stop_off":  stop_off,
                "total_off": total_off,
            })
        if fold_start_errs:
            fold_mae = (np.mean(fold_start_errs) + np.mean(fold_stop_errs)) / 2
            if fold_mae < best_overall_mae:
                best_overall_mae    = fold_mae
                best_config_for_run = config.copy()
                torch.save(model.state_dict(), ckpt_path)
                if scaler is not None:
                    joblib.dump(scaler, scaler_path)
        if os.path.exists(fold_ckpt):
            os.remove(fold_ckpt)
        
    report_df = pd.DataFrame(all_file_results)
    report_df.to_csv("full_file_performance.csv", index=False)
    best_5 = report_df.nsmallest(5, 'total_off')
    best_5.to_csv("top_5_best_files.csv", index=False)
    worst_5 = report_df.nlargest(5, 'total_off')
    worst_5.to_csv("top_5_worst_files.csv", index=False)
    mae_start  = np.mean(all_start_diffs)
    mae_stop   = np.mean(all_stop_diffs)
    std_start  = np.std(all_start_diffs)
    std_stop   = np.std(all_stop_diffs)
    total_std  = (std_start + std_stop) / 2 
    total_mae  = (mae_start + mae_stop) / 2
    print(f"\n--- Experiment: {config['note']} ---")
    print(f"Top 5 Worst Performers:\n{worst_5}")
    print(f"Top 5 Best Performers:\n{best_5}")
    print(f"\nFinal MAE Result -> Total: {total_mae:.2f} frames, std: {total_std:.2f}")
    print(f"Start: {mae_start:.2f} (±{std_start:.2f}) | Stop: {mae_stop:.2f} (±{std_stop:.2f})")

